<!-- ## IMPORTAZIONE LIBRERIE UTILI -->

# IMPORTAZIONE LIBRERIE

In [ ]:
import pandas as pd
import os

In [ ]:
os.getcwd()

In [ ]:
if not os.path.exists("raw_data"):
    os.mkdir("raw_data")

os.listdir()

# CREAZIONE e INFO DF_SITUAS

Creazione del df grezzo di SITUAS e osservazione delle informazioni generali

In [ ]:
os.listdir("fetching_data/")

In [ ]:
df_situas_raw = pd.read_csv("fetching_data/" + os.listdir("fetching_data/")[0], sep=';')

#Salvo il df grezzo
df_situas_raw.to_csv("raw_data/df_situas_raw.csv", index=False)

df_situas_raw.head()

In [ ]:
print(df_situas_raw.info())

In [ ]:
#Controllo Anno del dataset
df_situas_raw["Anno (Popolazione residente)"].value_counts() 

In [ ]:
df_situas_raw.describe().round()

# CREAZIONE e INFO DF_ISTAT

Creazione del df grezzo di ISTAT e osservazione delle informazioni generali

In [ ]:
df_istat_raw= pd.read_csv("fetching_data/incidenti_istat.csv")

df_istat_raw.to_csv("raw_data/df_istat_raw.csv", index=False)

df_istat_raw.head()

In [ ]:
print(df_istat_raw.info())

In [ ]:
df_istat_raw.describe().round()

# CLEANING DF_ISTAT
Pulizia dei dati grezzi e creazione di un df pulito

In [ ]:
print(df_istat_raw.info())
df_istat_raw.head()

In [ ]:
print(df_istat_raw["FREQ"].value_counts() ) #se FREQ è annuale per tutti potrei togliere la colonna
print('-------------------')
print(df_istat_raw["DATAFLOW"].value_counts())


In [ ]:
#CREO IL DATASET CON LE COLONNE DI INTERESSE CHE VERRA' PULITO
df_istat_clean=df_istat_raw[["FREQ","REF_AREA","DATA_TYPE","RESULT","TIME_PERIOD","OBS_VALUE"]]

df_istat_clean.head()


In [ ]:
print("Valori unici 'REF_AREA':" , df_istat_clean["REF_AREA"].nunique())
print('---------------------------')
print("Valori unici 'TIME_PERIOD':" , df_istat_clean["TIME_PERIOD"].nunique())
print("Anni registrati:" , df_istat_clean['TIME_PERIOD'].unique())  
print('---------------------------')
print("Numero di record tra incidenti (ROADACC) e morti/feriti (KILLINJ):",df_istat_clean["DATA_TYPE"].value_counts())
print('---------------------------')
print("Numero di record per variabile della colonna:",df_istat_clean["RESULT"].value_counts())
print("------------------------------")
print("Righe duplicate:" , df_istat_clean.duplicated().sum())

In [ ]:
df_istat_clean.groupby(["DATA_TYPE", "RESULT"])["OBS_VALUE"].describe()

In [ ]:
os.getcwd()
os.listdir()

In [ ]:
if not os.path.exists("df_clean"):
    os.mkdir("df_clean")

In [ ]:
df_istat_clean.to_csv("df_clean/df_istat_clean.csv",index=False)

# CLEANING DF_SITUAS
Pulizia dei dati grezzi e creazione di un df pulito

In [ ]:
print(df_situas_raw.info())
df_situas_raw.head()

In [ ]:
print("Valori nulli per colonna: ", df_situas_raw.isna().sum())
print('----------------------')
print("Record duplicati:" , df_situas_raw.duplicated().sum())

In [ ]:
print("Numero di regioni registrate: ",df_situas_raw["Codice Regione"].nunique())
print('-----------------------------')
print("Capoluoghi di regione registrati: ",df_situas_raw["Capoluogo di Regione"].nunique())
print('-----------------------------')
print("I codici mostrano che sono stati registrati n di codici alfanumerici dei comuni: ",df_situas_raw["Codice Comune (alfanumerico)"].nunique())
print("I codici mostrano che sono stati registrati n di codici numerici dei comuni: ",df_situas_raw["Codice Comune (numerico)"].nunique())
print('-----------------------------')
print("Anno (popolazione residente ) e n registrazioni: ",df_situas_raw["Anno (Popolazione residente)"].value_counts())
print("Anno (superficie) e n registrazioni: ",df_situas_raw["Anno (Superficie)"].value_counts())

In [ ]:
df_situas_clean = df_situas_raw[["Codice Provincia/Uts","Codice Comune (numerico)","Comune",
                                 "Popolazione residente","Superficie (Kmq)","Anno (Popolazione residente)"]]

df_situas_clean.head()

In [ ]:
print("Il numero univoco di comuni registrati è: ", df_situas_clean["Codice Comune (numerico)"].nunique())
print("---------------------------------------")
print("Il numero di nomi di comuni univoci (tenendo tutto in minuscolo in maniera che siano uguali) è: ", df_situas_clean["Comune"].str.lower().str.strip().nunique())
print("Il numero di nomi di comuni univoci è: ", df_situas_clean["Comune"].nunique())
print("N.B. Quest' ultima riga mi fa pensare che ci siano comuni con lo stesso nome, infatti ci sono più codici comuni che nomi") 
print("-----------------------------")

#Guardo se ci sono comuni omonimi
df_situas_clean[df_situas_clean["Comune"].duplicated(keep=False)]

In [ ]:
#cerco quale sia il numero che contiene sia . che ,
df_situas_clean[df_situas_clean["Superficie (Kmq)"].str.contains("\.", regex=True)]["Superficie (Kmq)"]

In [ ]:

df_situas_clean["Superficie (Kmq)"] = (df_situas_clean["Superficie (Kmq)"].str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
                                       .astype(float))

In [ ]:
df_situas_clean.info()

In [ ]:
#ho visto che c'è un comune che ha valore nullo, voglio capire che riga sia
df_situas_clean[df_situas_clean["Comune"].isna()]

In [ ]:
df_situas_clean["Comune"] = df_situas_clean["Comune"].fillna("None.")
df_situas_clean.isna().sum()

#per il comune (chiamato "None") che veniva letto come nullo sono andato a cercarlo tra gli elenchi disponibili e l' ho inserito, si chiama "None."
# https://www.istat.it/it/files/2020/12/C01.pdf --->n comuni 2020 
# https://www.istat.it/storage/ASI/2024/capitoli/C01.pdf -->n comuni 2024
df_situas_clean.info()

In [ ]:
df_situas_clean.to_csv ("df_clean/df_situas_clean.csv",index=False)

# JOIN DI DF_ISTAT_CLEAN E DF_SITUAS_CLEAN

In [ ]:
print(df_istat_clean["REF_AREA"].dtype)
print(df_situas_clean["Codice Comune (numerico)"].dtype)

In [ ]:
#Integro i dataset mantenendo però una colonna che mi indichi (indicator=True) quali valori siano comuni
df_final = pd.merge(df_istat_clean, df_situas_clean, left_on="REF_AREA", right_on="Codice Comune (numerico)", how="left", indicator=True)
df_final

In [ ]:
df_final[df_final["TIME_PERIOD"]== 2020]["REF_AREA"].nunique()

In [ ]:
df_final.to_csv("raw_data/df_final_raw.csv",index=False)

In [ ]:
print(df_final['_merge'].value_counts())
print("--------------------")
print(df_final.isnull().sum())


In [ ]:
comuni_no_match=df_final[df_final["_merge"]=="left_only"].groupby("REF_AREA").nunique()
comuni_no_match["_merge"].sum()


In [ ]:
len(comuni_no_match["Codice Comune (numerico)"])

In [ ]:
n_comuni_no_match=0

for i in comuni_no_match["Comune"]:
    n_comuni_no_match += 1

print(n_comuni_no_match)

    

Sono stati identificati 24.867 record relativi a 675 codici comunali non presenti nell'anagrafica SITUAS 2020. Tali record corrispondono verosimilmente a codici comunali modificati (per scopi amministrativi es. fusioni, soppressioni o altro) nel periodo 2001-2020.

In [ ]:
df_final_from_20_to_24_raw = df_final[(df_final["_merge"] == "both") & (df_final["TIME_PERIOD"] >= 2020)]

df_final_from_20_to_24_raw

In [ ]:
df_final_from_20_to_24_raw.to_csv("raw_data/df_final_from_20_to_24_raw.csv",index=False)

# CLEANING DF_FINAL

In [ ]:
print(df_final_from_20_to_24_raw.isna().sum())
print('----------------')
print(df_final_from_20_to_24_raw.info())

In [ ]:
#ricontrollo colonne che vorrei eliminare 
print(df_final_from_20_to_24_raw["FREQ"].value_counts())
print("------------------------------")
print(df_final_from_20_to_24_raw["Anno (Popolazione residente)"].value_counts())

In [ ]:
print(df_final_from_20_to_24_raw.duplicated().sum())
print("-------------------")
print(df_final_from_20_to_24_raw[["REF_AREA", "DATA_TYPE", "RESULT", "TIME_PERIOD"]].duplicated().sum())

In [ ]:
df_final_from_20_to_24_raw

In [ ]:
#elimino colonne che non mi servono
df_final_from_20_to_24 = df_final_from_20_to_24_raw.drop(columns=["FREQ","Codice Comune (numerico)",
                                                                  "Anno (Popolazione residente)",
                                                                  "_merge"])

In [ ]:
print(df_final_from_20_to_24.info())
df_final_from_20_to_24.head(3)

Il dataset SITUAS fornisce popolazione residente e superficie comunale riferite al 2020. Tali variabili sono state utilizzate come caratteristiche territoriali statiche per tutti gli anni considerati (2020-2024), osservando che in quegli anni ci sono state minime variazioni tra il numero di comuni o tra i residenti di questi ultimi.

In [ ]:
df_final_from_20_to_24["Codice Provincia/Uts"]=df_final_from_20_to_24["Codice Provincia/Uts"].astype(int)
df_final_from_20_to_24["Popolazione residente"]=df_final_from_20_to_24["Popolazione residente"].astype(int)

In [ ]:
print(df_final_from_20_to_24.info())
df_final_from_20_to_24.head(3)

In [ ]:
#Rinomino i valori per renderli più chiari
df_final_from_20_to_24["DATA_TYPE"] = df_final_from_20_to_24["DATA_TYPE"].replace({"KILLINJ": "morti_e_feriti",
                                                                                   "ROADACC": "incidenti_stradali"})

df_final_from_20_to_24["RESULT"] = df_final_from_20_to_24["RESULT"].replace({"M": "morti",
                                                                             "F": "feriti",
                                                                             "9": "incidenti"})

df_final_from_20_to_24=df_final_from_20_to_24.rename(columns={"REF_AREA" : "ID_COMUNE", "RESULT" : "RESULTS", 
                                                              "Codice Provincia/Uts" : "CODICE PROVINCIA", 
                                                              "Comune" : "COMUNE",
                                                              "Popolazione residente" : "RESIDENTI",
                                                              "Superficie (Kmq)" : "SUPERFICIE (KMQ)",})

df_final_from_20_to_24_clean=df_final_from_20_to_24
df_final_from_20_to_24_clean

In [ ]:
os.listdir()

In [ ]:
print(df_final_from_20_to_24_clean[["ID_COMUNE", "DATA_TYPE", "RESULTS", "TIME_PERIOD"]].duplicated().sum())

In [ ]:
df_final_from_20_to_24_clean.to_csv("df_clean/df_final_from_20_to_24_clean.csv",index=False)